# Enhanced Flash Crash Prediction Training Notebook
Improved GRU/LSTM training pipeline with:
- Better sequence generation
- Class imbalance handling
- Focal loss
- Early stopping
- ROC-AUC evaluation
- Model saving


## Install / Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


## Load Prepared Dataset

In [2]:
data = pd.read_csv("flash_crash_ready_dataset.csv")

print(data.shape)
data.head()

(234702, 17)


,Date,ticker,Open,High,Low,Close,Volume,VWAP,return,volatility,momentum,volume_change,vwap_diff,high_low_spread,open_close_return,turnover_change,crash_label
0,2007-12-11,ADANIPORTS,-0.072516,-0.075689,-0.081371,-0.085051,-0.310769,-0.077456,-0.973551,1.376206,0.024652,-0.051911,-1.874208,0.412591,-1.202954,-0.052548,0
1,2007-12-12,ADANIPORTS,-0.091463,-0.084847,-0.091184,-0.089251,-0.319921,-0.086699,-0.405895,0.810433,-0.264672,-0.042897,-0.627410,0.469612,0.221583,-0.044444,0
2,2007-12-13,ADANIPORTS,-0.088369,-0.052412,-0.085591,-0.053190,0.003825,-0.061475,3.276124,1.156212,0.261524,0.200725,2.024327,2.788292,3.505404,0.218506,0
3,2007-12-14,ADANIPORTS,-0.049742,-0.056228,-0.057781,-0.060720,-0.274494,-0.057427,-0.657854,1.255025,0.035254,-0.086290,-0.758106,-0.013398,-0.999947,-0.085737,0
4,2007-12-17,ADANIPORTS,-0.049703,-0.045543,-0.089025,-0.086367,-0.227913,-0.064057,-2.217551,1.561236,-0.183762,-0.012963,-5.381261,4.143309,-3.335218,-0.014680,0


## Define Feature Columns

In [3]:
features = [
"Open","High","Low","Close","Volume","VWAP",
"return","volatility","momentum","volume_change",
"vwap_diff","high_low_spread","open_close_return","turnover_change"
]

## Generate Time Sequences

In [4]:
sequence_length = 20

X = []
y = []

for ticker in data["ticker"].unique():
    
    stock_df = data[data["ticker"] == ticker].reset_index(drop=True)

    values = stock_df[features].values
    labels = stock_df["crash_label"].values

    n = len(values) - sequence_length

    if n <= 0:
        continue

    for i in range(n):
        X.append(values[i:i+sequence_length])
        y.append(labels[i+sequence_length])

X = np.array(X, dtype="float32")
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (233722, 20, 14)
y shape: (233722,)


## Train/Test Split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (186977, 20, 14)
Test: (46745, 20, 14)


## Handle Class Imbalance

In [6]:
from collections import Counter

counter = Counter(y_train)
print(counter)

class_weights = {
    0: 1,
    1: (counter[0] / counter[1])
}

print("Class weights:", class_weights)

Counter({np.int64(0): 185004, np.int64(1): 1973})
Class weights: {0: 1, 1: 93.76786619361378}


## Focal Loss Function

In [7]:
def focal_loss(gamma=2., alpha=.25):
    def loss(y_true, y_pred):
        bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
        pt = tf.exp(-bce)
        return alpha * (1-pt)**gamma * bce
    return loss

## Build Enhanced GRU Model

In [8]:
gru_model = Sequential([
    
    GRU(128, return_sequences=True, input_shape=(sequence_length, len(features))),
    Dropout(0.3),
    
    GRU(64, return_sequences=True),
    Dropout(0.3),
    
    GRU(32),
    
    Dense(16, activation="relu"),
    
    Dense(1, activation="sigmoid")
])

gru_model.compile(
    optimizer="adam",
    loss=focal_loss(),
    metrics=["accuracy"]
)

gru_model.summary()

c:\Users\kavan\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 20, 128)        │        55,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 20, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 20, 64)         │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 20, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ (None, 32)             │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 102,497 (400.38 KB)

 Trainable params: 102,497 (400.38 KB)

 Non-trainable params: 0 (0.00 B)

## Training Configuration

In [9]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

## Train GRU Model

In [10]:
history = gru_model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=128,
    validation_split=0.2,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/30
1169/1169 ━━━━━━━━━━━━━━━━━━━━ 68s 54ms/step - accuracy: 0.9142 - loss: 0.0357 - val_accuracy: 0.9418 - val_loss: 0.0161
Epoch 2/30
1169/1169 ━━━━━━━━━━━━━━━━━━━━ 64s 55ms/step - accuracy: 0.9338 - loss: 0.0324 - val_accuracy: 0.9481 - val_loss: 0.0123
Epoch 3/30
1169/1169 ━━━━━━━━━━━━━━━━━━━━ 60s 52ms/step - accuracy: 0.9379 - loss: 0.0310 - val_accuracy: 0.9264 - val_loss: 0.0158
Epoch 4/30
1169/1169 ━━━━━━━━━━━━━━━━━━━━ 64s 55ms/step - accuracy: 0.9390 - loss: 0.0312 - val_accuracy: 0.9413 - val_loss: 0.0161
Epoch 5/30
1169/1169 ━━━━━━━━━━━━━━━━━━━━ 64s 55ms/step - accuracy: 0.9419 - loss: 0.0301 - val_accuracy: 0.9346 - val_loss: 0.0148
Epoch 6/30
1169/1169 ━━━━━━━━━━━━━━━━━━━━ 65s 56ms/step - accuracy: 0.9427 - loss: 0.0295 - val_accuracy: 0.9348 - val_loss: 0.0128
Epoch 7/30
1169/1169 ━━━━━━━━━━━━━━━━━━━━ 58s 49ms/step - accuracy: 0.9424 - loss: 0.0294 - val_accuracy: 0.9609 - val_loss: 0.0116
Epoch 8/30
1169/1169 ━━━━━━━━━━━━━━━━━━━━ 46s 39ms/step - accuracy: 0.9460 -

## Evaluate Model

In [11]:
pred_probs = gru_model.predict(X_test)

pred = (pred_probs > 0.2).astype(int)

print(classification_report(y_test, pred))

print("ROC AUC:", roc_auc_score(y_test, pred_probs))

1461/1461 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step
              precision    recall  f1-score   support

           0       1.00      0.35      0.52     46273
           1       0.02      0.99      0.03       472

    accuracy                           0.36     46745
   macro avg       0.51      0.67      0.27     46745
weighted avg       0.99      0.36      0.51     46745

ROC AUC: 0.9680749234370667


## Save Model

In [12]:
gru_model.save("enhanced_flash_crash_model.keras")

print("Model saved successfully.")

Model saved successfully.


## Optional: Train LSTM Model

In [13]:
lstm_model = Sequential([
    
    LSTM(128, return_sequences=True, input_shape=(sequence_length, len(features))),
    Dropout(0.3),
    
    LSTM(64),
    
    Dense(16, activation="relu"),
    
    Dense(1, activation="sigmoid")
])

lstm_model.compile(
    optimizer="adam",
    loss=focal_loss(),
    metrics=["accuracy"]
)

lstm_model.summary()

c:\Users\kavan\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 20, 128)        │        73,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 20, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 123,681 (483.13 KB)

 Trainable params: 123,681 (483.13 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
lstm_model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=128,
    validation_split=0.2,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/30
1169/1169 ━━━━━━━━━━━━━━━━━━━━ 71s 57ms/step - accuracy: 0.9147 - loss: 0.0369 - val_accuracy: 0.9279 - val_loss: 0.0178
Epoch 2/30
1169/1169 ━━━━━━━━━━━━━━━━━━━━ 71s 61ms/step - accuracy: 0.9387 - loss: 0.0309 - val_accuracy: 0.9437 - val_loss: 0.0135
Epoch 3/30
1169/1169 ━━━━━━━━━━━━━━━━━━━━ 59s 51ms/step - accuracy: 0.9397 - loss: 0.0304 - val_accuracy: 0.9354 - val_loss: 0.0159
Epoch 4/30
1169/1169 ━━━━━━━━━━━━━━━━━━━━ 59s 50ms/step - accuracy: 0.9394 - loss: 0.0296 - val_accuracy: 0.9417 - val_loss: 0.0141
Epoch 5/30
1169/1169 ━━━━━━━━━━━━━━━━━━━━ 58s 50ms/step - accuracy: 0.9428 - loss: 0.0290 - val_accuracy: 0.9585 - val_loss: 0.0106


## Save LSTM Model

In [15]:
lstm_model.save("enhanced_flash_crash_lstm.keras")